# Mixed turbulence: KT, D6, and adaptive transport

Run all cells after downloading the small exports from Vera. Only the mixed-turbulence comparison is included in the current batch. All methods retain the same explicit diffusion. Differences between methods are not errors against an exact solution.


In [ ]:
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt

root=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"runs").is_dir() and (p/".git").exists())
methods=["kt","d6","adaptive"]
colors={"kt":"tab:blue","d6":"tab:orange","adaptive":"tab:green"}
def data_dir(case,method): return root/"runs"/f"experiment-{case}-{method}"/"data"
def time_series(path):
    with path.open() as stream: names=re.findall(r"[A-Za-z][A-Za-z0-9_]*",stream.readline())
    rows=np.atleast_2d(np.loadtxt(path))
    if rows.shape[1]!=len(names): raise ValueError(f"Unexpected header in {path}")
    return dict(zip(names,rows.T))
for case in ["turbulence"]:
    ready=[m for m in methods if (data_dir(case,m)/"export_complete.txt").exists()]
    print(case, "exports ready:", ", ".join(ready) or "none yet")


## How often KT is selected

The fraction includes the two-face buffer, counts positive faces in active directions, and is sampled at the first RK stage. It is not a fraction of physical energy dissipated. Fixed KT uses fraction one by definition; fixed D6 uses zero.


In [ ]:
fig,ax=plt.subplots(figsize=(7,3.5))
for ax,case in [(ax,"turbulence")]:
    for m in methods:
        data=data_dir(case,m)
        path=data/"transport_usage.dat"
        if path.exists():
            a=np.atleast_2d(np.loadtxt(path))
            if a.size: ax.plot(a[:,1],a[:,4],label=m,color=colors[m])
        elif m in ["kt","d6"] and (data/"time_series.dat").exists():
            ts=time_series(data/"time_series.dat")
            ax.plot(ts["t"],np.full_like(ts["t"],1.0 if m=="kt" else 0.0),label=m,color=colors[m])
    ax.set_title(case); ax.set_xlabel("Time"); ax.set_ylim(-.02,1.02)
    if ax.lines: ax.legend()
ax.set_ylabel("Fraction of faces selecting KT")
fig.tight_layout()
plt.show()


## Turbulence: separate physical-velocity spectra

The exporter uses the Fourier Helmholtz projection: longitudinal velocity is parallel to k (compressional), transverse velocity is perpendicular (vortical). Power is one half the squared velocity per spherical shell, not relativistic kinetic energy and not automatically the normalization of the Wick-code E_N/E_L. The legend includes the actual last snapshot time. Compare curves at matching times; different final times can otherwise bias the comparison.


In [ ]:
spectra={}
fig,axes=plt.subplots(1,2,figsize=(12,4))
for m in methods:
    path=data_dir("turbulence",m)/"velocity_components.dat"
    if not path.exists(): continue
    a=np.atleast_2d(np.loadtxt(path)); times=np.unique(a[:,0]); last=a[a[:,0]==times[-1]]
    spectra[m]=a
    for ax,col in zip(axes,[3,4]):
        good=(last[:,1]>0)&(last[:,col]>0)
        ax.loglog(last[good,1],last[good,col],label=f"{m}, t={times[-1]:.4g}",color=colors[m])
for ax,title in zip(axes,["Vortical velocity power","Compressional velocity power"]):
    ax.set_title(title); ax.set_xlabel("Wavenumber k"); ax.set_ylabel("Velocity power per shell")
    if ax.lines: ax.legend()
fig.tight_layout(); plt.show()
if spectra:
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for m,a in spectra.items():
        for t in np.unique(a[:,0]):
            snapshot=a[a[:,0]==t]
            for ax,col in zip(axes,[3,4]):
                good=(snapshot[:,1]>0)&(snapshot[:,col]>0)
                ax.loglog(snapshot[good,1],snapshot[good,col],color=colors[m],alpha=.25)
        for ax in axes: ax.plot([],[],color=colors[m],label=m)
    for ax,title in zip(axes,["Vortical spectrum evolution","Compressional spectrum evolution"]):
        ax.set_title(title); ax.set_xlabel("Wavenumber k"); ax.set_ylabel("Velocity power per shell"); ax.legend()
    fig.tight_layout(); plt.show()


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.5))
for m in methods:
    data=data_dir("turbulence",m)
    path=data/"velocity_component_summary.dat"
    if path.exists():
        a=np.atleast_2d(np.loadtxt(path))
        axes[0].plot(a[:,0],a[:,2],label=m,color=colors[m])
        axes[1].plot(a[:,0],a[:,3],label=m,color=colors[m])
        print(m,"maximum Parseval relative error:",a[:,-1].max())
    path=data/"time_series.dat"
    if path.exists():
        ts=time_series(path)
        if "ekin" in ts: axes[2].plot(ts["t"],ts["ekin"],label=m,color=colors[m])
for ax,title in zip(axes,["Integrated vortical velocity power","Integrated compressional velocity power","Pencil kinetic-energy diagnostic"]):
    ax.set_title(title); ax.set_xlabel("Time")
    if ax.lines: ax.legend()
fig.tight_layout(); plt.show()


At one resolution these plots reveal differences, not a ranking of true accuracy. Check physical admissibility and stability, smooth-wave behavior, and shock oscillations before interpreting turbulence spectra. Convergence needs matched initial conditions across resolutions. More surviving vortical power may indicate reduced damping, but can also include numerical noise.
